# Train Bi-Encoder MiniLM tren AllNLI pair 500k

Notebook nay moi la ban dung de fine-tune tren tap AllNLI `pair` lon.

- Train dataset: `sentence-transformers/all-nli`, subset `pair`, split `train`.
- So mau train mac dinh: `500000`.
- Loss: `MultipleNegativesRankingLoss`.
- Evaluation trong luc train: `sentence-transformers/all-nli`, subset `pair-score`, split `dev`.
- Benchmark cuoi: `phdquang/allnli-pair-class-processed` de so voi TF-IDF va pretrained MiniLM.

Tren Kaggle hay bat **GPU** va **Internet** truoc khi chay.

## 1. Cai thu vien

In [ ]:
%pip install -q -U "datasets>=3.0" "sentence-transformers>=5.0,<6" "accelerate>=1.0" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=15.0"

## 2. Import va cau hinh

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, average_precision_score, precision_recall_curve, precision_recall_fscore_support, roc_auc_score

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TRAIN_DATASET_ID = "sentence-transformers/all-nli"
BENCHMARK_DATASET_ID = "phdquang/allnli-pair-class-processed"

OUTPUT_ROOT = Path("/kaggle/working/allnli-pair-500k-minilm-biencoder")
FINAL_MODEL_DIR = OUTPUT_ROOT / "final"
RESULTS_DIR = OUTPUT_ROOT / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_TRAIN_SAMPLES = 500_000
EVAL_PAIR_SCORE_SAMPLES = 10_000
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 128
EVAL_STEPS = 1000
SAVE_STEPS = 1000
MAX_RETRIEVAL_QUERIES = 1000
RETRIEVAL_POOL_SIZE = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU chua duoc bat. Trong Kaggle: Settings > Accelerator > GPU, roi restart session.")

print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_ROOT)

## 3. Load AllNLI pair de train 500k mau

Day la phan quan trong: khong dung `pair-class` de train nua. `pair-class` chi de benchmark cuoi.

In [ ]:
raw_pair_train = load_dataset(TRAIN_DATASET_ID, "pair", split="train")
print(raw_pair_train)
print(raw_pair_train.column_names)
print(raw_pair_train[0])

train_pair = raw_pair_train.shuffle(seed=SEED)
if MAX_TRAIN_SAMPLES and len(train_pair) > MAX_TRAIN_SAMPLES:
    train_pair = train_pair.select(range(MAX_TRAIN_SAMPLES))

required = {"anchor", "positive"}
missing = required - set(train_pair.column_names)
if missing:
    raise ValueError(f"Dataset pair thieu cot: {missing}")

remove_columns = [col for col in train_pair.column_names if col not in ["anchor", "positive"]]
if remove_columns:
    train_pair = train_pair.remove_columns(remove_columns)

train_pair = train_pair.filter(lambda row: bool(str(row["anchor"]).strip()) and bool(str(row["positive"]).strip()))
print("So cap train thuc te:", len(train_pair))

## 4. Load pair-score/dev de evaluate trong luc train

In [ ]:
pair_score_dev = load_dataset(TRAIN_DATASET_ID, "pair-score", split="dev")
pair_score_dev = pair_score_dev.shuffle(seed=SEED)
if EVAL_PAIR_SCORE_SAMPLES and len(pair_score_dev) > EVAL_PAIR_SCORE_SAMPLES:
    pair_score_dev = pair_score_dev.select(range(EVAL_PAIR_SCORE_SAMPLES))

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=[str(x) for x in pair_score_dev["sentence1"]],
    sentences2=[str(x) for x in pair_score_dev["sentence2"]],
    scores=[float(x) for x in pair_score_dev["score"]],
    batch_size=EVAL_BATCH_SIZE,
    main_similarity="cosine",
    name="allnli-pair-score-dev",
    show_progress_bar=True,
)
print("Eval pair-score rows:", len(pair_score_dev))

## 5. Load benchmark pair-class cua project

Doan load dataset nay dung theo yeu cau cua ban. No dung de so sanh model sau train, khong dung lam train set chinh.

In [ ]:
from datasets import load_dataset

ds = load_dataset("phdquang/allnli-pair-class-processed")

train_ds = ds["train"]
dev_ds = ds["dev"]
test_ds = ds["test"]

print(ds)
print(train_ds.column_names)
print(train_ds[0])

TEXT_A = "premise_clean" if "premise_clean" in train_ds.column_names else "premise"
TEXT_B = "hypothesis_clean" if "hypothesis_clean" in train_ds.column_names else "hypothesis"
train_df = train_ds.to_pandas()
dev_df = dev_ds.to_pandas()
test_df = test_ds.to_pandas()

if "label_name" not in dev_df.columns:
    label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
    for frame in [train_df, dev_df, test_df]:
        frame["label_name"] = frame["label"].map(label_map)

## 6. Ham benchmark chung

In [ ]:
POSITIVE_LABEL = "entailment"
NEGATIVE_RETRIEVAL_LABEL = "contradiction"

def choose_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    index = int(np.argmax(f1))
    return float(thresholds[index]), float(f1[index])

def classification_metrics(y_true, scores, threshold):
    predictions = scores >= threshold
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, predictions, average="binary", zero_division=0)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }

def transformer_pair_scores(model, frame):
    left = model.encode(frame[TEXT_A].fillna("").astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    right = model.encode(frame[TEXT_B].fillna("").astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    return np.sum(left * right, axis=1)

def transformer_retrieval_metrics(model, frame, seed):
    positives = frame[frame["label_name"] == POSITIVE_LABEL].reset_index(drop=True)
    negatives = frame[frame["label_name"] == NEGATIVE_RETRIEVAL_LABEL].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    if len(positives) > MAX_RETRIEVAL_QUERIES:
        positives = positives.iloc[rng.choice(len(positives), MAX_RETRIEVAL_QUERIES, replace=False)].reset_index(drop=True)
    queries = model.encode(positives[TEXT_B].astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    relevant = model.encode(positives[TEXT_A].astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    distractors_all = model.encode(negatives[TEXT_A].astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    ranks = []
    for index, query in enumerate(queries):
        distractor_ids = rng.choice(len(distractors_all), RETRIEVAL_POOL_SIZE - 1, replace=False)
        scores = np.concatenate(([float(relevant[index] @ query)], distractors_all[distractor_ids] @ query))
        permutation = rng.permutation(RETRIEVAL_POOL_SIZE)
        relevant_position = int(np.flatnonzero(permutation == 0)[0])
        ranking = np.argsort(-scores[permutation], kind="stable")
        ranks.append(int(np.flatnonzero(ranking == relevant_position)[0]) + 1)
    ranks = np.asarray(ranks)
    return {
        "queries": int(len(ranks)),
        "precision_at_1": float(np.mean(ranks <= 1)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "mrr": float(np.mean(1.0 / ranks)),
        "mean_rank": float(np.mean(ranks)),
    }

def evaluate_transformer(name, model):
    dev_scores = transformer_pair_scores(model, dev_df)
    test_scores = transformer_pair_scores(model, test_df)
    dev_targets = (dev_df["label_name"] == POSITIVE_LABEL).to_numpy()
    test_targets = (test_df["label_name"] == POSITIVE_LABEL).to_numpy()
    threshold, _ = choose_threshold(dev_targets, dev_scores)
    return {
        "model": name,
        **classification_metrics(test_targets, test_scores, threshold),
        **transformer_retrieval_metrics(model, test_df, SEED + 1),
    }, test_scores

## 7. TF-IDF baseline tren benchmark

In [ ]:
tfidf = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), max_features=50000, min_df=2, sublinear_tf=True, norm="l2")
tfidf.fit(pd.concat([train_df[TEXT_A], train_df[TEXT_B]], ignore_index=True).fillna(""))

def tfidf_scores(frame):
    left = tfidf.transform(frame[TEXT_A].fillna(""))
    right = tfidf.transform(frame[TEXT_B].fillna(""))
    return np.asarray(left.multiply(right).sum(axis=1)).ravel()

dev_targets = (dev_df["label_name"] == POSITIVE_LABEL).to_numpy()
test_targets = (test_df["label_name"] == POSITIVE_LABEL).to_numpy()
tfidf_dev_scores = tfidf_scores(dev_df)
tfidf_test_scores = tfidf_scores(test_df)
tfidf_threshold, _ = choose_threshold(dev_targets, tfidf_dev_scores)
tfidf_result = {"model": "TF-IDF", **classification_metrics(test_targets, tfidf_test_scores, tfidf_threshold)}
tfidf_result

## 8. Pretrained MiniLM baseline

In [ ]:
model = SentenceTransformer(BASE_MODEL, device="cuda")
model.max_seq_length = MAX_SEQ_LENGTH
pretrained_result, pretrained_test_scores = evaluate_transformer("Pretrained MiniLM", model)
pd.DataFrame([tfidf_result, pretrained_result])

## 9. Fine-tune tren AllNLI pair 500k

In [ ]:
bf16_supported = bool(hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_ROOT / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=not bf16_supported,
    bf16=bf16_supported,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

loss = MultipleNegativesRankingLoss(model)
trainer = SentenceTransformerTrainer(model=model, args=training_args, train_dataset=train_pair, loss=loss, evaluator=evaluator)
trainer.train()

## 10. Luu model va benchmark fine-tuned

In [ ]:
model.save_pretrained(str(FINAL_MODEL_DIR))
final_eval = evaluator(model, output_path=str(RESULTS_DIR))
finetuned_result, finetuned_test_scores = evaluate_transformer("Fine-tuned MiniLM pair-500k", model)

comparison = pd.DataFrame([tfidf_result, pretrained_result, finetuned_result])
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

predictions = test_df[["premise", "hypothesis", "label_name"]].copy()
predictions["tfidf_cosine"] = tfidf_test_scores
predictions["pretrained_minilm_cosine"] = pretrained_test_scores
predictions["finetuned_minilm_pair_500k_cosine"] = finetuned_test_scores
predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

metadata = {
    "train_dataset": f"{TRAIN_DATASET_ID}/pair",
    "benchmark_dataset": BENCHMARK_DATASET_ID,
    "base_model": BASE_MODEL,
    "requested_train_samples": MAX_TRAIN_SAMPLES,
    "actual_train_samples": len(train_pair),
    "epochs": NUM_EPOCHS,
    "batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_seq_length": MAX_SEQ_LENGTH,
    "gpu": torch.cuda.get_device_name(0),
    "final_eval": {str(k): float(v) if hasattr(v, "item") else v for k, v in final_eval.items()},
}
(RESULTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

comparison

## 11. Nen output de tai ve

In [ ]:
model_zip = shutil.make_archive("/kaggle/working/allnli-pair-500k-minilm-biencoder-model", "zip", FINAL_MODEL_DIR)
results_zip = shutil.make_archive("/kaggle/working/allnli-pair-500k-minilm-biencoder-results", "zip", RESULTS_DIR)
print("Model ZIP:", model_zip)
print("Results ZIP:", results_zip)

## 12. Tuy chon push len Hugging Face

In [ ]:
PUSH_TO_HUB = False
HUB_MODEL_ID = "phdquang/allnli-pair-500k-minilm-biencoder"
HUB_PRIVATE = True

if PUSH_TO_HUB:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    model.push_to_hub(HUB_MODEL_ID, token=token, private=HUB_PRIVATE, exist_ok=True)
    print("Pushed:", f"https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("Skip push_to_hub")